# 📊 SBI証券 ポートフォリオ投資判断システム
**スマートフォンからSBIのスクリーンショットをアップロードするだけで投資判断を自動生成します**

### 使い方
1. 上メニューの ▶ ですべてのセルを順番に実行
2. セル3でスクリーンショットをアップロード
3. 最終セルに投資判断レポートが表示される

In [ ]:
# ── セル1: 必要パッケージのインストール ──────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install('easyocr')
install('yfinance')
install('Pillow')
print('✅ インストール完了')

In [ ]:
# ── セル2: 分析エンジン定義 ──────────────────────────────────
import math, re, time, os
import yfinance as yf
from datetime import datetime, timedelta
from IPython.display import display, HTML

# ── テクニカル指標（scanner_v12継承）──

def _ema(data, period):
    k = 2 / (period + 1)
    e = [data[0]]
    for v in data[1:]:
        e.append(v * k + e[-1] * (1 - k))
    return e

def _calc_rsi(closes, period=14):
    if len(closes) < period + 2: return 50.0
    gains, losses = [], []
    for i in range(1, len(closes)):
        d = closes[i] - closes[i-1]
        gains.append(max(d, 0.0))
        losses.append(max(-d, 0.0))
    ag = sum(gains[:period]) / period
    al = sum(losses[:period]) / period
    for i in range(period, len(gains)):
        ag = (ag * (period-1) + gains[i]) / period
        al = (al * (period-1) + losses[i]) / period
    return 100.0 if al == 0 else 100 - 100 / (1 + ag / al)

def _calc_indicators(closes, highs, lows, volumes):
    if len(closes) < 90: return None
    ef = _ema(closes, 9); es = _ema(closes, 26)
    ml = [f - s for f, s in zip(ef, es)]
    sl = _ema(ml[25:], 6)
    hist_t = ml[-1] - sl[-1]; hist_p = ml[-2] - sl[-2]
    k14r = []
    for i in range(13, len(closes)):
        hi = max(highs[i-13:i+1]); lo = min(lows[i-13:i+1])
        k14r.append(50 if hi == lo else (closes[i]-lo)/(hi-lo)*100)
    sm14 = _ema(k14r, 3)
    ma25 = sum(closes[-25:])/25; ma25p = sum(closes[-26:-1])/25
    rsi  = _calc_rsi(closes[-30:])
    ma20 = sum(closes[-20:])/20
    std20= math.sqrt(sum((c-ma20)**2 for c in closes[-20:])/20)
    bb_lo= ma20 - 2*std20; bb_hi = ma20 + 2*std20
    bb_pos=(closes[-1]-bb_lo)/(bb_hi-bb_lo) if bb_hi>bb_lo else 0.5
    vol_ma20 = sum(volumes[-20:])/20
    vol_ratio= volumes[-1]/vol_ma20 if vol_ma20>0 else 1.0
    ma5  = sum(closes[-5:])/5; ma5p = sum(closes[-6:-1])/5
    ma75 = sum(closes[-75:])/75
    return {
        'hist_t': hist_t, 'hist_chg': hist_t-hist_p,
        'k14': sm14[-1], 'k14_chg': sm14[-1]-sm14[-2],
        'dev': (closes[-1]-ma25)/ma25*100,
        'rsi': rsi, 'bb_pos': bb_pos, 'vol_ratio': vol_ratio,
        'ma5_slope': (ma5-ma5p)/ma5p*100,
        'dev75': (closes[-1]-ma75)/ma75*100,
        'price': closes[-1],
        'support': min(lows[-60:]), 'resistance': max(highs[-60:]),
        'price_change_5d': (closes[-1]-closes[-6])/closes[-6]*100 if len(closes)>=6 else 0,
        'price_change_20d': (closes[-1]-closes[-21])/closes[-21]*100 if len(closes)>=21 else 0,
    }

def fetch_ohlcv(code, days=130):
    ticker = f'{code}.T'
    end = datetime.now(); start = end - timedelta(days=days)
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if df is None or len(df) < 90: return None
        return {'closes': df['Close'].tolist(), 'highs': df['High'].tolist(),
                'lows': df['Low'].tolist(), 'volumes': df['Volume'].tolist()}
    except: return None

SECTOR_MAP = {
    '1605':('エネルギー','原油・天然ガス開発'),
    '1662':('エネルギー','石油資源開発'),
    '2782':('小売','100円ショップ'),
    '3901':('素材','難燃防護服・機能素材'),
    '4245':('環境','水処理・排水設備'),
    '4550':('医薬品','製薬（住友化学系）'),
    '3983':('素材','鉄鋼加工・金属資材'),
    '9602':('エンタメ','映画・不動産'),
    '9876':('小売','ファッション・アパレル'),
}

def judge_position(pos, ind):
    pnl = pos.get('pnl_pct', 0) or 0
    code = pos['code']
    sector, detail = SECTOR_MAP.get(code, ('不明', ''))

    def tech_s():
        if not ind: return ''
        parts = []
        if ind['rsi'] <= 30:   parts.append(f"RSI{ind['rsi']:.0f}(売られ過ぎ)")
        elif ind['rsi'] >= 70: parts.append(f"RSI{ind['rsi']:.0f}(買われ過ぎ)")
        else:                   parts.append(f"RSI{ind['rsi']:.0f}")
        if ind['bb_pos'] < 0.2: parts.append('BB下限近傍')
        elif ind['bb_pos'] > 0.8: parts.append('BB上限近傍')
        if ind['hist_chg'] > 0: parts.append('MACD改善')
        elif ind['hist_chg'] < 0: parts.append('MACD悪化')
        return '｜'.join(parts) + '。' if parts else ''

    if pnl <= -12:
        return ('即売却（損切）','★★★',
                f'損益率{pnl:.1f}%は信用取引の危機水域。追証リスク回避のため即座に損切り実行せよ。',
                '🔴 HIGH')
    if pnl <= -7:
        return ('即売却（損切）','★★★',
                f'損益率{pnl:.1f}%は損切りライン完全突破。損失拡大前の撤退が鉄則。',
                '🔴 HIGH')
    if pnl <= -5:
        if ind and ind['bb_pos']<0.15 and ind['rsi']<30:
            return ('保有継続（反発監視）','★★',
                    f'損益{pnl:.1f}%だが売られ過ぎ。{tech_s()}翌日確認後に再判断。','🟠 MEDIUM')
        return ('損切り','★★',
                f'損益{pnl:.1f}%。テクニカル反発根拠なし。{tech_s()}{detail}の回復見込み薄。',
                '🟠 MEDIUM')
    if sector == 'エネルギー' and pnl > 0:
        if ind and ind['rsi'] > 65:
            return ('一部利確検討','★★',
                    f'損益{pnl:+.1f}%。{tech_s()}RSI過熱。利益50%確定を推奨。','🟡 LOW-MED')
        return ('保有継続','★★',
                f'損益{pnl:+.1f}%。{tech_s()}エネルギー安保テーマで支持継続。WTI60ドル割れが損切りトリガー。',
                '🟢 LOW')
    if ind and ind['k14']<=30 and ind['k14_chg']>=1 and ind['rsi']<=40 and ind['bb_pos']<=0.3 and pnl>-3:
        return ('追加買い候補','★★★',
                f'進化買いシグナル発動。K14={ind["k14"]:.0f}↑ / RSI{ind["rsi"]:.0f} / BB下位。{tech_s()}',
                '🟢 LOW')
    if ind and ind['rsi']>=55 and ind['ma5_slope']>0.3 and ind['hist_chg']>0 and pnl>0:
        return ('保有継続（上昇継続）','★★',
                f'損益{pnl:+.1f}%。{tech_s()}上昇モメンタム継続。利益を伸ばす局面。','🟢 LOW')
    if ind and ind['bb_pos']<0.2 and ind['rsi']<35 and pnl>-5:
        return ('保有継続（反発待ち）','★★',
                f'売られ過ぎ。{tech_s()}短期反発確率高。-5%到達で即損切り。','🟡 LOW-MED')
    if -3 <= pnl <= 1:
        return ('保有継続（様子見）','★',
                f'損益{pnl:+.1f}%で方向感なし。{tech_s()}-5%で即損切り発動。','🟡 LOW-MED')
    if pnl > 1:
        return ('保有継続','★★',f'損益{pnl:+.1f}%。{tech_s()}上昇余地を継続観察。','🟢 LOW')
    return ('損切り検討','★',f'損益{pnl:+.1f}%。{tech_s()}明確な反発根拠なし。','🟠 MEDIUM')

print('✅ エンジン定義完了')

In [ ]:
# ── セル3: SBIスクリーンショットをアップロードしてOCR解析 ────
# ※ スマートフォンからコラボを使う場合: ファイルアイコン→アップロードでも可

from google.colab import files
import easyocr
import re
from pathlib import Path

print('📸 SBIスクリーンショットをアップロードしてください...')
uploaded = files.upload()

if not uploaded:
    print('アップロードなし → サンプルデータ（2026/04/29実績）を使用')
    # 直近スクリーンショット実績（手動フォールバック）
    POSITIONS = [
        {'code':'1605','name':'INPEX',        'current_price':4123, 'avg_cost':4058.00,'pnl_pct':+1.52,'pnl_yen': None},
        {'code':'1662','name':'石油資源開発',  'current_price':2295, 'avg_cost':2269.70,'pnl_pct':+1.04,'pnl_yen': None},
        {'code':'2782','name':'セリア',        'current_price':3440, 'avg_cost':3555.00,'pnl_pct':-3.33,'pnl_yen':-11845},
        {'code':'3901','name':'アゼアス',      'current_price': 633, 'avg_cost': 684.00,'pnl_pct':-7.78,'pnl_yen':-30677},
        {'code':'4245','name':'ダイキアクシス','current_price': 703, 'avg_cost': 739.00,'pnl_pct':-5.13,'pnl_yen': -7596},
        {'code':'4550','name':'住友ファーマ',  'current_price':1851, 'avg_cost':2164.50,'pnl_pct':-14.80,'pnl_yen':-32145},
        {'code':'3983','name':'モリテック',    'current_price': 234, 'avg_cost': 232.86,'pnl_pct': -0.17,'pnl_yen':  -285},
        {'code':'9602','name':'東宝',          'current_price':1461, 'avg_cost':1465.00,'pnl_pct': -0.32,'pnl_yen':  -462},
        {'code':'9876','name':'コックス',      'current_price': 238, 'avg_cost': 270.00,'pnl_pct':-12.09,'pnl_yen': -9817},
    ]
else:
    img_path = list(uploaded.keys())[0]
    print(f'✅ アップロード完了: {img_path}')

    # OCR実行
    reader = easyocr.Reader(['ja','en'], gpu=False, verbose=False)
    raw    = reader.readtext(img_path)
    print(f'OCR: {len(raw)} ブロック検出')

    # パターンで価格・損益率を抽出
    RE_PRICE = re.compile(r'(\d{1,5}[,，]\d{3}(?:\.\d{1,2})?|\d{2,6}(?:\.\d{1,2})?)(?:円)?')
    RE_PCT   = re.compile(r'([+\-−]?\d{1,3}(?:\.\d{1,2})?)(?:%|％)')
    RE_CODE  = re.compile(r'\b(\d{4})\b')
    KNOWN_NAMES = {
        '1605':'INPEX','1662':'石油資源開発','2782':'セリア',
        '3901':'アゼアス','4245':'ダイキアクシス','4550':'住友ファーマ',
        '3983':'モリテック','9602':'東宝','9876':'コックス',
    }

    def bbox_y(bbox): return sum(p[1] for p in bbox) / len(bbox)
    sorted_raw = sorted(raw, key=lambda r: bbox_y(r[0]))

    POSITIONS = []
    pending   = None
    prev_y    = None

    for bbox, text, conf in sorted_raw:
        y = bbox_y(bbox)
        cm = RE_CODE.search(text)
        if cm:
            code = cm.group(1)
            if 1000 <= int(code) <= 9999:
                if pending and pending.get('current_price'): POSITIONS.append(pending)
                pending = {'code':code,'name':KNOWN_NAMES.get(code,text[:8]),
                           'current_price':None,'avg_cost':None,'pnl_pct':None,'pnl_yen':None}
                prev_y = y; continue
        if not pending: continue
        prices = [float(p.replace(',','').replace('，','')) for p in RE_PRICE.findall(text)
                  if float(p.replace(',','').replace('，','')) > 50]
        if prices and pending['current_price'] is None:
            if len(prices) >= 2:
                pending['current_price'] = prices[0]; pending['avg_cost'] = prices[1]
            elif prices:
                pending['current_price'] = prices[0]
            continue
        pcts = RE_PCT.findall(text)
        if pcts and pending['pnl_pct'] is None:
            v = float(pcts[-1].replace('−','-').replace('＋','+'))
            pending['pnl_pct'] = v

    if pending and pending.get('current_price'): POSITIONS.append(pending)
    for p in POSITIONS:
        if p['pnl_pct'] is None and p['current_price'] and p['avg_cost']:
            p['pnl_pct'] = (p['current_price']-p['avg_cost'])/p['avg_cost']*100

    if not POSITIONS:
        print('⚠️ OCR解析失敗。以下に手動でデータを入力してください。')

print(f'\n検出ポジション: {len(POSITIONS)}件')
for p in POSITIONS:
    sign = '+' if (p['pnl_pct'] or 0) >= 0 else ''
    print(f"  {p['code']} {p['name']:12s}  現在値{p['current_price']:>8,.0f}円  "
          f"平均{p['avg_cost']:>9,.2f}円  損益率{sign}{p['pnl_pct'] or 0:.2f}%")

In [ ]:
# ── セル4: 全銘柄テクニカル分析 ─────────────────────────────
RESULTS = []
print(f'[分析] {len(POSITIONS)} 銘柄を取得中...\n')
for i, pos in enumerate(POSITIONS, 1):
    code = pos['code']; name = pos['name']
    print(f'  [{i}/{len(POSITIONS)}] {code} {name}', end=' ... ', flush=True)
    data = fetch_ohlcv(code)
    ind  = _calc_indicators(data['closes'],data['highs'],data['lows'],data['volumes']) if data else None
    action, stars, reason, risk = judge_position(pos, ind)
    sector, detail = SECTOR_MAP.get(code, ('不明',''))
    RESULTS.append({'code':code,'name':name,'sector':sector,'detail':detail,
                    'cur':pos['current_price'],'avg':pos['avg_cost'],
                    'pnl_pct':pos['pnl_pct'] or 0,'pnl_yen':pos.get('pnl_yen'),
                    'action':action,'stars':stars,'reason':reason,'risk':risk,
                    'indicators':ind})
    print(action)
    time.sleep(0.3)

print('\n✅ 分析完了')

In [ ]:
# ── セル5: 投資判断レポート表示 ─────────────────────────────

ACTION_COLOR = {
    '即売却（損切）':'#d32f2f','損切り':'#e53935','損切り検討':'#f57c00',
    '一部利確検討':'#f9a825',
    '保有継続（反発監視）':'#1565c0','保有継続（反発待ち）':'#1976d2',
    '保有継続（上昇継続）':'#2e7d32','保有継続（様子見）':'#546e7a',
    '保有継続':'#2e7d32','追加買い候補':'#1b5e20','様子見':'#78909c',
}
ACTION_ICON = {
    '即売却（損切）':'🚨','損切り':'⛔','損切り検討':'⚠️','一部利確検討':'💰',
    '保有継続（反発監視）':'👁','保有継続（反発待ち）':'⏳','保有継続（上昇継続）':'🚀',
    '保有継続（様子見）':'⏸','保有継続':'✅','追加買い候補':'🎯','様子見':'⏸',
}
ORDER = {'即売却（損切）':0,'損切り':1,'損切り検討':2,'一部利確検討':3,
         '保有継続（反発監視）':4,'保有継続（反発待ち）':5,'追加買い候補':6,
         '保有継続（上昇継続）':7,'保有継続':8,'保有継続（様子見）':9,'様子見':10}

sorted_r = sorted(RESULTS, key=lambda r: ORDER.get(r['action'], 99))

cards_html = ''
for r in sorted_r:
    act   = r['action']
    color = ACTION_COLOR.get(act, '#607d8b')
    icon  = ACTION_ICON.get(act, '')
    pc    = '#c62828' if r['pnl_pct']>0 else '#1565c0'
    sign  = '+' if r['pnl_pct']>=0 else ''
    ind   = r['indicators']
    ind_h = ''
    if ind:
        ind_h = (f"<div style='font-size:11px;color:#546e7a;background:#f5f5f5;padding:5px 8px;"
                 f"border-radius:5px;margin:6px 0;display:flex;gap:10px;flex-wrap:wrap'>"
                 f"<span>RSI:<b>{ind['rsi']:.0f}</b></span>"
                 f"<span>BB位置:<b>{ind['bb_pos']:.2f}</b></span>"
                 f"<span>K14:<b>{ind['k14']:.0f}</b></span>"
                 f"<span>MA乖離:<b>{ind['dev']:+.1f}%</b></span>"
                 f"<span>出来高比:<b>{ind['vol_ratio']:.1f}x</b></span>"
                 f"<span>5D騰落:<b>{ind['price_change_5d']:+.1f}%</b></span></div>")
    cards_html += f"""
<div style='background:#fff;border-radius:10px;padding:14px;margin-bottom:10px;
            box-shadow:0 1px 4px rgba(0,0,0,.12);border-left:5px solid {color}'>
  <div style='display:flex;justify-content:space-between;align-items:flex-start;
              padding-bottom:8px;border-bottom:1px solid #e0e0e0;margin-bottom:8px'>
    <div>
      <span style='font-size:17px;font-weight:700'>{r['code']}</span>
      <span style='font-size:14px;color:#424242;margin-left:6px'>{r['name']}</span>
      <span style='font-size:11px;background:#e8eaf6;color:#3949ab;border-radius:4px;
                   padding:2px 6px;margin-left:5px'>{r['sector']}</span>
    </div>
    <div style='text-align:right'>
      <span style='font-size:19px;font-weight:700;color:{pc}'>{sign}{r['pnl_pct']:.2f}%</span>
      <div style='font-size:11px;color:#757575'>現在{r['cur']:,.1f} / 平均{r['avg']:,.2f}</div>
    </div>
  </div>
  <div style='margin-bottom:8px'>
    <span style='background:{color};color:#fff;padding:5px 12px;border-radius:6px;
                 font-size:13px;font-weight:700'>{icon} {act}</span>
    <span style='margin-left:10px;font-size:14px'>{r['stars']}</span>
    <span style='margin-left:8px;font-size:12px'>{r['risk']}</span>
  </div>
  {ind_h}
  <div style='font-size:13px;line-height:1.6;color:#333'>{r['reason']}</div>
</div>"""

urgent = sum(1 for r in RESULTS if '即売却' in r['action'] or r['action']=='損切り')
add_b  = sum(1 for r in RESULTS if '追加買い' in r['action'])
hold   = sum(1 for r in RESULTS if '保有継続' in r['action'])

html_out = f"""
<div style='font-family:-apple-system,sans-serif;max-width:680px;margin:0 auto'>
<h2 style='color:#1a237e;margin-bottom:4px'>📊 投資判断レポート</h2>
<div style='font-size:12px;color:#757575;margin-bottom:12px'>生成: {datetime.now().strftime('%Y/%m/%d %H:%M')} ｜ 信用建玉対象</div>
<div style='display:flex;gap:8px;margin-bottom:16px;flex-wrap:wrap'>
  <span style='background:#d32f2f;color:#fff;padding:5px 12px;border-radius:20px;font-size:13px;font-weight:600'>🚨 即損切り: {urgent}件</span>
  <span style='background:#2e7d32;color:#fff;padding:5px 12px;border-radius:20px;font-size:13px;font-weight:600'>✅ 保有継続: {hold}件</span>
  <span style='background:#1b5e20;color:#fff;padding:5px 12px;border-radius:20px;font-size:13px;font-weight:600'>🎯 追加買い: {add_b}件</span>
</div>
{cards_html}
<div style='font-size:11px;color:#9e9e9e;margin-top:12px;text-align:center'>
本レポートは参考情報です。投資判断はご自身の責任で行ってください。
</div>
</div>"""

display(HTML(html_out))